In [ ]:
import os
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
import segmentation_models_pytorch as smp
from tqdm import tqdm
import pandas as pd
import cv2
from pycocotools.coco import COCO
from albumentations.pytorch import ToTensorV2

In [ ]:
# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
IMAGE_SIZE = 1024


def canny_edge_aug(image):
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 100)  # lower thresholds = more sensitive
    overlay = image.copy()
    overlay[edges > 0] = [0, 255, 0]  # green overlay for edges
    return A.Compose([A.Normalize(), ToTensorV2()])(image=overlay)["image"]



def hsv_with_edges_aug(image):
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    edges = cv2.Canny(hsv[:, :, 2], 100, 200)
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    rgb[edges > 0] = [255, 0, 255]  # magenta edges
    return A.Compose([A.Normalize(), ToTensorV2()])(image=rgb)["image"]


def harris_corner_aug(image):
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = np.float32(gray)
    corners = cv2.cornerHarris(gray, 2, 3, 0.04)
    corners = cv2.dilate(corners, None)
    image[corners > 0.01 * corners.max()] = [255, 0, 0]
    return A.Compose([A.Normalize(), ToTensorV2()])(image=image)["image"]

def stretch_contrast_individual(image):
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    stretched = np.zeros_like(image, dtype=np.uint8)

    for i in range(3):
        c = image[:, :, i].astype(np.float32)
        min_val, max_val = np.min(c), np.max(c)
        if max_val > min_val:
            c_stretched = (c - min_val) * 255.0 / (max_val - min_val)
            stretched[:, :, i] = np.clip(c_stretched, 0, 255).astype(np.uint8)
        else:
            stretched[:, :, i] = np.full_like(c, 127)  # fallback to neutral gray

    return A.Compose([A.Normalize(), ToTensorV2()])(image=stretched)["image"]



def histogram_equalization(image):
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    img_eq = np.empty_like(image)
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))  # gentle adjustment

    for i in range(3):
        img_eq[:, :, i] = clahe.apply(image[:, :, i])
        
    return A.Compose([A.Normalize(), ToTensorV2()])(image=img_eq)["image"]

def custom_aug_wrap(fn):
    def wrapped(image, mask=None):
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        if mask is not None:
            mask = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
        else:
            mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)

        transformed_image = fn(image)

        # Ensure mask is float32 and normalized to [0,1], shape [1, H, W]
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        if mask_tensor.max() > 1:
            mask_tensor /= 255.0

        return {
            "image": transformed_image,
            "mask": mask_tensor
        }
    return wrapped


# Final dictionary
augmentations = {
    "basic_resize_only": A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE), A.Normalize(), ToTensorV2()
    ]),
    "random_rotate_90": A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE), A.RandomRotate90(p=1.0), A.Normalize(), ToTensorV2()
    ]),
    "brightness_contrast": A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.RandomBrightnessContrast(brightness_limit=(0.0, 0.2), contrast_limit=(0.0, 0.2), p=0.7),
        A.Normalize(), ToTensorV2()
    ]),
    "clahe_brightness": A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE), A.CLAHE(p=0.5),
        A.RandomBrightnessContrast(p=0.5), A.Normalize(), ToTensorV2()
    ]),
    "shift_scale_rotate": A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Affine(translate_percent=0.05, scale=1.0, rotate=10, p=0.7),
        A.Normalize(), ToTensorV2()
    ]),
    "gaussian_blur_flip": A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.GaussianBlur(blur_limit=3, p=0.4),
        A.Normalize(), ToTensorV2()
    ]),
    "comprehensive_heavy_aug": A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.5), A.HueSaturationValue(p=0.3),
        A.Affine(translate_percent=0.05, scale=1.0, rotate=10, p=0.5),
        A.Normalize(), ToTensorV2()
    ]),
    "canny_edge": custom_aug_wrap(canny_edge_aug),
    "hsv_edges": custom_aug_wrap(hsv_with_edges_aug),
    "harris_corners": custom_aug_wrap(harris_corner_aug),
    "stretch_contrast": custom_aug_wrap(stretch_contrast_individual),
    "histogram_equalization": custom_aug_wrap(histogram_equalization)
}



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import glob

# Update to your actual size
IMAGE_SIZE = 1024

# Load 10 sample images
image_paths = sorted(glob.glob("/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/train/*.jpg"))[:3]


# Show all 10 augmentations on sample images
def show_all_augmentations(image_paths, augmentations):
    for aug_name, transform in augmentations.items():
        print(f"\n🔍 Showing {aug_name}")
        for i, img_path in enumerate(image_paths):
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            orig_img = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))

            aug_img_tensor = transform(image=orig_img)["image"]
            aug_img = aug_img_tensor.permute(1, 2, 0).cpu().numpy()
            aug_img = np.clip(aug_img, 0, 1)

            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.imshow(orig_img)
            plt.title("Original")
            plt.axis("off")

            plt.subplot(1, 2, 2)
            plt.imshow(aug_img)
            plt.title(f"Augmented: {aug_name}")
            plt.axis("off")
            plt.tight_layout()
            plt.show()

# Run
show_all_augmentations(image_paths, augmentations)


In [ ]:
from torch.utils.data import Dataset
import numpy as np
import cv2
from pycocotools.coco import COCO
import torch

class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform=None):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_info = self.coco.loadImgs(self.image_ids[idx])[0]
        image_path = f"{self.img_dir}/{image_info['file_name']}"
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        ann_ids = self.coco.getAnnIds(imgIds=image_info['id'])
        anns = self.coco.loadAnns(ann_ids)

        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Ensure mask has shape [1, H, W]
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)  # [H, W] -> [1, H, W]
        elif mask.ndim == 3 and mask.shape[0] != 1:
            mask = mask[0].unsqueeze(0)  # [H, W] for first channel only

        return image, mask.float()


In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import segmentation_models_pytorch as smp
import torch.nn as nn

# Choose device
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

# Paths
train_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/train"
train_ann_path = os.path.join(train_img_dir, "_annotations.coco.json")
val_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/valid"
val_ann_path = os.path.join(val_img_dir, "_annotations.coco.json")

# Combined loss: Dice + BCEWithLogits
class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=1.0, bce_weight=1.0):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary')
        self.bce = nn.BCEWithLogitsLoss()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight

    def forward(self, inputs, targets):
        return self.dice_weight * self.dice(inputs, targets) + self.bce_weight * self.bce(inputs, targets)

# Metrics
def dice_coef(preds, targets, threshold=0.5, eps=1e-6):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean()

def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()

In [ ]:
# Prepare to store results
results = []

# Loop over all augmentations
for aug_name, train_transform in augmentations.items():
    print(f"\n🔁 Training with: {aug_name}")

    train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path, transform=train_transform)
    val_dataset = COCOSegmentationDataset(val_img_dir, val_ann_path, transform=augmentations["basic_resize_only"])
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

    model = smp.Unet(
        encoder_name="efficientnet-b3",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    loss_fn = CombinedLoss().to(device)

    best_dice = 0
    best_model_path = f"best_trans_model_{aug_name}.pth"

    for epoch in range(20):
        model.train()
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
            images, masks = images.to(device).float(), masks.to(device).float()
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()

        model.eval()
        val_dice, val_iou, val_loss = 0, 0, 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
                images, masks = images.to(device).float(), masks.to(device).float()
                outputs = model(images)
                val_loss += loss_fn(outputs, masks).item()
                val_dice += dice_coef(outputs, masks).item()
                val_iou += iou_score(outputs, masks).item()

        val_dice /= len(val_loader)
        val_iou /= len(val_loader)
        val_loss /= len(val_loader)

        scheduler.step(val_loss)

        # Print metrics
        print(f"📊 Epoch {epoch+1} | Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}")

        # Save best model
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), best_model_path)
            print(f"✅ Saved best model for {aug_name} at epoch {epoch+1}")


    

    results.append({
        "augmentation": aug_name,
        "val_dice": round(val_dice, 4),
        "val_iou": round(val_iou, 4),
        "val_loss": round(val_loss, 4)
    })

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

# Prepare results and resume training from 'canny_edge'
results = []
augment_keys = list(augmentations.keys())
start_index = augment_keys.index("canny_edge")

for aug_name in augment_keys[start_index:]:
    train_transform = augmentations[aug_name]
    print(f"\n🔁 Training with: {aug_name}")

    train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path, transform=train_transform)
    val_dataset = COCOSegmentationDataset(val_img_dir, val_ann_path, transform=augmentations["basic_resize_only"])

    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    model = smp.Unet(
        encoder_name="efficientnet-b3",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    loss_fn = CombinedLoss().to(device)

    best_dice = 0
    for epoch in range(20):
        model.train()
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
            if callable(train_transform):
                images = images.to(device).float()
                masks = masks.to(device).float()
            else:
                images, masks = images.to(device).float(), masks.to(device).float()

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()

        model.eval()
        val_dice, val_iou, val_loss = 0, 0, 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
                images, masks = images.to(device).float(), masks.to(device).float()
                outputs = model(images)
                val_loss += loss_fn(outputs, masks).item()
                val_dice += dice_coef(outputs, masks).item()
                val_iou += iou_score(outputs, masks).item()

        val_dice /= len(val_loader)
        val_iou /= len(val_loader)
        val_loss /= len(val_loader)

        print(f"📊 Epoch {epoch+1} | Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f} | IoU: {val_iou:.4f}")
        scheduler.step(val_loss)

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), f"best_trans_model_{aug_name}.pth")
            print(f"✅ Saved best model for {aug_name} at epoch {epoch+1}")

    results.append({
        "augmentation": aug_name,
        "val_dice": round(val_dice, 4),
        "val_iou": round(val_iou, 4),
        "val_loss": round(val_loss, 4)
    })



In [ ]:
results_df = pd.DataFrame(results).sort_values(by="val_dice", ascending=False)
print(results_df)


In [ ]:
results